In [1]:
import pandas as pd 
from transformers import T5Tokenizer , Trainer , TrainingArguments , T5ForConditionalGeneration # models used to perform different tasks 

In [2]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [3]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:
train_data.shape

(14732, 3)

In [5]:
val_data.shape

(818, 3)

In [6]:
# random sampling 
train_data = train_data.sample(n=4000 , random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500 , random_state=42).reset_index(drop=True)

In [7]:
# data preprocessing 

import re 

def clean_data(text):
    text = re.sub(r"\r\n" , " " , text) # lines 
    text = re.sub(r"\s+" , " " , text) # spaces 
    text = re.sub(r"<.*?>" , " " , text) # html tags <h1>
    text = text.strip().lower()
    return text 

In [8]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

### TOKENIZER 

In [9]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [10]:
def tokenize(data):
    inputs = tokenizer(data["dialogue"] , padding="max_length" , max_length=512 , truncation=True)
    targets = tokenizer(data["summary"] , padding="max_length" , max_length=150 , truncation=True)

    inputs["labels"] = targets["input_ids"]  # token ids => add to input as labels 
    return inputs

In [11]:
train_dataset = train_data.apply(tokenize , axis=1).tolist() # converted to list cause its compataible
val_dataset = val_data.apply(tokenize , axis=1).tolist() 

In [12]:
train_dataset[5]
# input_ids = dialogue =>token ids 
# 1 => EOS , 0 => padding 
# attention_mask 
# labels - target = summary token 

{'input_ids': [2576, 83, 10, 5722, 25, 55, 3, 27341, 10, 3, 58, 3947, 15, 10, 69, 2287, 7, 43, 118, 18454, 58, 2576, 83, 10, 24, 31, 7, 269, 55, 3, 27341, 10, 3, 23, 410, 77, 31, 17, 214, 5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

### WORKING WITH OUR MODEL 

In [13]:
# NLP = generation task (conditional = dependent on input )

model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [14]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

In [15]:
training_args = TrainingArguments(
    output_dir = "./results", # to save best models 

    num_train_epochs = 6, # number of epochs 
    weight_decay = 0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500 # when our lr will start reaching that particular value (0.01) / 0 -> lr
)
    

In [16]:
trainer = Trainer(
    model = model , 
    args = training_args,
    train_dataset = train_dataset , 
    eval_dataset = val_dataset
)

In [17]:
# training 
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.631273,0.379161
2,0.396137,0.360677
3,0.373064,0.355531
4,0.362364,0.351454
5,0.354644,0.350468
6,0.351133,0.349856


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

AttributeError: 'TrainOutput' object has no attribute 'to'

In [19]:
model.save_pretrained("./saved_summarization_model")
tokenizer.save_pretrained("./saved_summarization_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summarization_model\\tokenizer_config.json',
 './saved_summarization_model\\tokenizer.json')

In [24]:
model= T5ForConditionalGeneration.from_pretrained("./saved_summarization_model")
tokenizer= T5Tokenizer.from_pretrained("./saved_summarization_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [34]:
# Testing 

def summarize_dialogue(dialogue):
    # clean
    dialogue = clean_data(dialogue)

    # tokenize 
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # summary create => input ids(tokens)
    model.to(device)
    targets = model.generate(
        input_ids= inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    # summary => decode 
    summary = tokenizer.decode(targets[0] , skip_special_tokens=True)
    return summary


In [35]:
test_dialogue = """
Rahul: Hi Priya! Did you read the latest article about Artificial Intelligence?

Priya: Hey Rahul! Yes, I did. It’s fascinating how AI is transforming our daily lives.

Rahul: Absolutely. From personal assistants like Siri and Alexa to complex data analysis, AI is everywhere these days.

Priya: True. Even in our college projects, using AI tools has made things much easier. I’m particularly impressed with how AI is being used in healthcare.

Rahul: That’s right. AI can help in early diagnosis of diseases and even assist doctors during surgeries. I heard some hospitals use AI for patient monitoring as well.

Priya: But don’t you think there are some concerns? For instance, people worry about job losses and privacy issues.

Rahul: That’s a valid point. Automation can reduce employment in some sectors, but it also creates new opportunities. As for privacy, there should be regulations to ensure data is protected.

Priya: Agreed. The key is to use AI responsibly. I’m excited to see how AI continues to evolve in the future.

Rahul: Me too! Maybe we should take an AI course together next semester.

Priya: That’s a great idea. Let’s do it!"""

summary = summarize_dialogue(test_dialogue)
print("summary :", summary)

summary : rahul has read the latest article about artificial intelligence. priya is impressed with how ai is being used in healthcare. ai can help in early diagnosis of diseases and assist doctors during surgeries.


In [36]:
import os
print(os.getcwd())

C:\Users\VICTUS\prime\TRANSFORMER
